# Setup

In [8]:
# importing libraries 
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    RobustScaler
)

from src.data.preprocessing import UNSWPreprocessor

In [2]:
# notebook config
PROJECT_ROOT = Path.cwd().resolve().parents[1]

sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"

TRAIN_PATH = DATA_DIR / "UNSW_NB15_training-set.csv"
TEST_PATH = DATA_DIR / "UNSW_NB15_testing-set.csv"

print("Project root:", PROJECT_ROOT)
print("Train path:", TRAIN_PATH)
print("Test path:", TEST_PATH)

Project root: C:\Projects\Aeges-Q
Train path: C:\Projects\Aeges-Q\data\raw\UNSW_NB15_training-set.csv
Test path: C:\Projects\Aeges-Q\data\raw\UNSW_NB15_testing-set.csv


In [3]:
# load dataset
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

Training shape: (82332, 45)
Testing shape: (175341, 45)


In [5]:
# preprocessor 
preprocessor = UNSWPreprocessor()

X_train, y_train, attack_train = (
    preprocessor.split_features_and_target(train_df)
)

X_test, y_test, attack_test = (
    preprocessor.split_features_and_target(test_df)
)

In [15]:
preprocessor.identify_feature_types(X_train)

numerical_features = preprocessor.numerical_features
categorical_features = preprocessor.categorical_features

print(len(numerical_features))
print(len(categorical_features))

print(categorical_features)

39
3
['proto', 'service', 'state']


In [6]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\nTarget distribution:")
print(y_train.value_counts(normalize=True))

X_train: (82332, 42)
X_test: (175341, 42)

Target distribution:
label
1    0.5506
0    0.4494
Name: proportion, dtype: float64


# Preprocessing Strategy

Based on the exploratory data analysis, the UNSW-NB15 dataset requires
a preprocessing pipeline that addresses categorical encoding, large
differences in feature scale, highly skewed numerical distributions,
and potential feature redundancy.

Rather than applying all transformations universally, multiple
preprocessing variants will be constructed and evaluated according to
their suitability for different downstream models.

# Preprocessing Variant B — Standard Scaling

The EDA revealed large differences in numerical feature ranges. While
tree-based models are generally insensitive to feature scaling, scaling
is important for distance-based, margin-based, and quantum machine
learning models.

This variant applies standardization to numerical features while
preserving the existing categorical preprocessing strategy.

In [12]:
# scaled numerical pipeline
scaled_numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [13]:
# categorical pipeline 
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [16]:
# scaled preprocessor 
scaled_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            scaled_numeric_transformer,
            numerical_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [17]:
X_train_scaled = scaled_preprocessor.fit_transform(X_train)

X_test_scaled = scaled_preprocessor.transform(X_test)

In [18]:
print("Scaled train shape:", X_train_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)

Scaled train shape: (82332, 190)
Scaled test shape: (175341, 190)


In [19]:
scaled_feature_names = (
    scaled_preprocessor.get_feature_names_out()
)

In [20]:
numeric_feature_indices = [
    i
    for i, feature_name in enumerate(scaled_feature_names)
    if feature_name.startswith("numeric__")
]

In [21]:
X_train_scaled_numeric = (
    X_train_scaled[:, numeric_feature_indices]
)

In [22]:
numeric_means = np.asarray(
    X_train_scaled_numeric.mean(axis=0)
).ravel()

numeric_stds = np.sqrt(
    np.asarray(
        X_train_scaled_numeric.power(2).mean(axis=0)
    ).ravel()
    - numeric_means ** 2
)

In [23]:
print("Maximum absolute mean:", np.abs(numeric_means).max())

print("Minimum std:", numeric_stds.min())
print("Maximum std:", numeric_stds.max())

Maximum absolute mean: 1.5724755515983489e-13
Minimum std: 0.9999999999982245
Maximum std: 1.0000000000011098
